# 12 — P2: IBP for Vision Transformer

**Plan 2 — Phase 3.** Sound interval bound propagation through every operation
in the ViT-Tiny encoder, returning a logit interval `[l, u] ∈ R^{C}` for an
L∞ input box of radius `ε` around `x₀`.

## Per-component IBP rules
| Op | IBP rule |
|---|---|
| `Linear(W, b)` | `l' = W⁺·l + W⁻·u + b`, `u' = W⁺·u + W⁻·l + b` |
| `Conv2d` (stride=kernel) | same as Linear after `unfold` (we use a manual reshape) |
| `ReLU` | `[max(0,l), max(0,u)]` |
| `add(a, b)` (residual) | `[l_a+l_b, u_a+u_b]` |
| `x²` | `[a², b²]` if `0∉[a,b]` else `[0, max(a²,b²)]` |
| `mean` over dim | linear average of intervals |
| `1/√(t)` (`t > 0`) | monotone decreasing → `[1/√u_t, 1/√l_t]` |
| `a·b` (bilinear) | range = `[min, max]` of the four corners |
| `softmax` (naive sound) | `l_i = exp(l_i)/(exp(l_i)+Σ_{j≠i}exp(u_j))`, `u_i = exp(u_i)/(exp(u_i)+Σ_{j≠i}exp(l_j))` |

**Softmax note.** Plan 2 §3 calls for the Wei et al. 2023 convex softmax bounds
(arxiv 2303.01713) for tightness. Here we use the simpler naive interval bound,
which is sound but looser. This is the conservative (sound) interpretation; a
future iteration can swap in Wei et al. without changing any consumer.

In [9]:
!pip install -q numpy torch torchvision tqdm pyyaml

In [10]:
from __future__ import annotations
import math, json, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as transforms

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1234); np.random.seed(1234)
print(f'Device: {device}')

Device: cuda


In [11]:
# ── ViT-Tiny class (must match notebooks 09/10/11) ────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__(); self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)

class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=2):
        super().__init__(); h = embed_dim * mlp_ratio
        self.fc1, self.fc2 = nn.Linear(embed_dim, h), nn.Linear(h, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); x = x + self.mlp(self.norm2(x)); return x

class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
                                     for _ in range(num_layers)])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

In [12]:
# ── Interval bound primitives ─────────────────────────────────────────────
@dataclass
class Interval:
    """Element-wise interval [l, u].  l and u must be broadcastable."""
    l: torch.Tensor
    u: torch.Tensor
    def __post_init__(self):
        assert self.l.shape == self.u.shape, f'{self.l.shape} vs {self.u.shape}'
        assert torch.all(self.l <= self.u + 1e-6), 'lower exceeds upper'
    @property
    def shape(self): return self.l.shape
    def detach(self): return Interval(self.l.detach(), self.u.detach())


def ibp_linear(iv: Interval, W: torch.Tensor, b: torch.Tensor | None) -> Interval:
    """y = x @ W^T + b   →   element-wise interval via W+/W- decomposition."""
    Wp, Wn = W.clamp(min=0), W.clamp(max=0)
    # Treat all but last dim as batch
    l_out = iv.l @ Wp.T + iv.u @ Wn.T
    u_out = iv.u @ Wp.T + iv.l @ Wn.T
    if b is not None: l_out, u_out = l_out + b, u_out + b
    return Interval(l_out, u_out)


def ibp_relu(iv: Interval) -> Interval:
    return Interval(iv.l.clamp(min=0), iv.u.clamp(min=0))


def ibp_add_const(iv: Interval, c: torch.Tensor) -> Interval:
    return Interval(iv.l + c, iv.u + c)


def ibp_add(a: Interval, b: Interval) -> Interval:
    return Interval(a.l + b.l, a.u + b.u)


def ibp_square(iv: Interval) -> Interval:
    """x²: convex.  If 0 ∈ [l,u], lower=0 else lower = min(l²,u²).  upper = max(l²,u²)."""
    l2, u2 = iv.l ** 2, iv.u ** 2
    contains_zero = (iv.l <= 0) & (iv.u >= 0)
    lo = torch.where(contains_zero, torch.zeros_like(iv.l), torch.minimum(l2, u2))
    hi = torch.maximum(l2, u2)
    return Interval(lo, hi)


def ibp_mean_lastdim(iv: Interval) -> Interval:
    return Interval(iv.l.mean(dim=-1, keepdim=True), iv.u.mean(dim=-1, keepdim=True))


def ibp_inv_sqrt(iv: Interval, eps: float) -> Interval:
    """1/√(x+ε) on x ≥ 0 — monotone decreasing → swap bounds."""
    assert torch.all(iv.l >= -1e-6), 'inv_sqrt input not non-negative'
    lo_in = iv.l.clamp(min=0) + eps
    hi_in = iv.u.clamp(min=0) + eps
    return Interval(1.0 / torch.sqrt(hi_in), 1.0 / torch.sqrt(lo_in))


def ibp_mul(a: Interval, b: Interval) -> Interval:
    """
    Element-wise product of two intervals.
    Range = [min, max] of the four corner products.  a and b must be
    element-wise broadcastable to the same shape.
    """
    al, au = torch.broadcast_tensors(a.l, b.l)[0], torch.broadcast_tensors(a.u, b.u)[0]
    bl, bu = torch.broadcast_tensors(b.l, a.l)[0], torch.broadcast_tensors(b.u, a.u)[0]
    c1, c2 = a.l * b.l, a.l * b.u
    c3, c4 = a.u * b.l, a.u * b.u
    lo = torch.minimum(torch.minimum(c1, c2), torch.minimum(c3, c4))
    hi = torch.maximum(torch.maximum(c1, c2), torch.maximum(c3, c4))
    return Interval(lo, hi)


def ibp_matmul_sum(iv_a: Interval, iv_b: Interval, sum_dim: int = -1) -> Interval:
    """
    Sound IBP for a sum of bilinear products: out[i] = Σ_d a[i,d] · b[i,d]
    via summing per-element interval products.  Used for inner-product-style
    contractions where both inputs are intervals.
    """
    prod = ibp_mul(iv_a, iv_b)
    return Interval(prod.l.sum(dim=sum_dim), prod.u.sum(dim=sum_dim))


def ibp_softmax_last(iv: Interval) -> Interval:
    """
    Naive sound IBP for softmax on the last dim:
        l_i = exp(l_i) / ( exp(l_i) + Σ_{j≠i} exp(u_j) )
        u_i = exp(u_i) / ( exp(u_i) + Σ_{j≠i} exp(l_j) )
    Numerically stabilised by subtracting the row-wise max upper bound from
    every entry (does not change softmax).
    """
    # stability shift: subtract the upper-bound max along last dim
    m = iv.u.max(dim=-1, keepdim=True).values
    l_s, u_s = iv.l - m, iv.u - m
    e_l, e_u = torch.exp(l_s), torch.exp(u_s)
    sum_eu = e_u.sum(dim=-1, keepdim=True)
    sum_el = e_l.sum(dim=-1, keepdim=True)
    # denom for l_i: e_l_i + (sum_eu - e_u_i)
    denom_l = e_l + (sum_eu - e_u)
    denom_u = e_u + (sum_el - e_l)
    lo = e_l / denom_l
    hi = e_u / denom_u
    # numerical clamp to [0, 1] — softmax is provably in this range
    return Interval(lo.clamp(0, 1), hi.clamp(0, 1))

In [13]:
# ── IBP for ViT components ────────────────────────────────────────────────
def ibp_patch_embed(iv: Interval, conv: nn.Conv2d) -> Interval:
    """Conv2d with stride==kernel — equivalent to a linear map per-patch."""
    Wp = conv.weight.clamp(min=0)
    Wn = conv.weight.clamp(max=0)
    bias = conv.bias
    s = (conv.stride, conv.padding) if False else None
    l_out = F.conv2d(iv.l, Wp, None, conv.stride, conv.padding) \
          + F.conv2d(iv.u, Wn, None, conv.stride, conv.padding)
    u_out = F.conv2d(iv.u, Wp, None, conv.stride, conv.padding) \
          + F.conv2d(iv.l, Wn, None, conv.stride, conv.padding)
    if bias is not None:
        l_out = l_out + bias.view(1, -1, 1, 1)
        u_out = u_out + bias.view(1, -1, 1, 1)
    # flatten H,W into token dim:  (B, E, H', W') → (B, N, E)
    l_out = l_out.flatten(2).transpose(1, 2)
    u_out = u_out.flatten(2).transpose(1, 2)
    return Interval(l_out, u_out)


def ibp_rmsnorm(iv: Interval, norm: RMSNorm) -> Interval:
    """
    y_i = γ_i · x_i / √(mean(x²) + ε)
    """
    sq      = ibp_square(iv)                      # element-wise x²
    msq     = ibp_mean_lastdim(sq)                # (..., 1)
    inv_rms = ibp_inv_sqrt(msq, norm.eps)         # (..., 1)
    # x_i · (1/rms) — element broadcast over last dim
    inv_rms_b = Interval(inv_rms.l.expand_as(iv.l), inv_rms.u.expand_as(iv.u))
    z = ibp_mul(iv, inv_rms_b)
    # · γ_i (a constant scalar per channel)
    gamma = norm.weight
    gp, gn = gamma.clamp(min=0), gamma.clamp(max=0)
    out_l = z.l * gp + z.u * gn
    out_u = z.u * gp + z.l * gn
    return Interval(out_l, out_u)


def ibp_linear_proj(iv: Interval, lin: nn.Linear) -> Interval:
    return ibp_linear(iv, lin.weight, lin.bias)


def ibp_mhsa(iv: Interval, attn: MHSA) -> Interval:
    """
    Sound IBP through full attention block.
    iv shape: (B, N, E)
    """
    B, N, E = iv.shape
    H, D = attn.num_heads, attn.head_dim

    # 1. Q, K, V projections (linear)
    Q = ibp_linear_proj(iv, attn.W_q)             # (B, N, E)
    K = ibp_linear_proj(iv, attn.W_k)
    V = ibp_linear_proj(iv, attn.W_v)

    # reshape to (B, H, N, D)
    def to_heads(x): return x.view(B, N, H, D).permute(0, 2, 1, 3).contiguous()
    Qh = Interval(to_heads(Q.l), to_heads(Q.u))
    Kh = Interval(to_heads(K.l), to_heads(K.u))
    Vh = Interval(to_heads(V.l), to_heads(V.u))

    # 2. scores[h,i,j] = (1/√D) Σ_d Q[h,i,d] · K[h,j,d]
    # Expand to (B, H, N, N, D) and reduce over last dim via interval product.
    Q_e = Interval(Qh.l.unsqueeze(3), Qh.u.unsqueeze(3))       # (B,H,N,1,D)
    K_e = Interval(Kh.l.unsqueeze(2), Kh.u.unsqueeze(2))       # (B,H,1,N,D)
    Q_b = Interval(Q_e.l.expand(B, H, N, N, D), Q_e.u.expand(B, H, N, N, D))
    K_b = Interval(K_e.l.expand(B, H, N, N, D), K_e.u.expand(B, H, N, N, D))
    scores = ibp_matmul_sum(Q_b, K_b, sum_dim=-1)              # (B,H,N,N)
    s_l = scores.l * attn.scale
    s_u = scores.u * attn.scale
    scores = Interval(s_l, s_u)

    # 3. softmax over last dim (j)
    A = ibp_softmax_last(scores)                                # (B,H,N,N)

    # 4. out[h,i,d] = Σ_j A[h,i,j] · V[h,j,d]
    A_e = Interval(A.l.unsqueeze(-1), A.u.unsqueeze(-1))        # (B,H,N,N,1)
    V_e = Interval(Vh.l.unsqueeze(2), Vh.u.unsqueeze(2))        # (B,H,1,N,D)
    A_b = Interval(A_e.l.expand(B, H, N, N, D), A_e.u.expand(B, H, N, N, D))
    V_b = Interval(V_e.l.expand(B, H, N, N, D), V_e.u.expand(B, H, N, N, D))
    out = ibp_matmul_sum(A_b, V_b, sum_dim=3)                   # sum over j → (B,H,N,D)

    # 5. concat heads + output projection
    out = Interval(out.l.permute(0, 2, 1, 3).contiguous().view(B, N, E),
                   out.u.permute(0, 2, 1, 3).contiguous().view(B, N, E))
    return ibp_linear_proj(out, attn.W_o)


def ibp_mlp_block(iv: Interval, mlp: MLPBlock) -> Interval:
    iv = ibp_linear_proj(iv, mlp.fc1)
    iv = ibp_relu(iv)
    return ibp_linear_proj(iv, mlp.fc2)


def ibp_block(iv: Interval, blk: TransformerBlock) -> Interval:
    n1 = ibp_rmsnorm(iv, blk.norm1)
    a  = ibp_mhsa(n1, blk.attn)
    iv = ibp_add(iv, a)
    n2 = ibp_rmsnorm(iv, blk.norm2)
    m  = ibp_mlp_block(n2, blk.mlp)
    return ibp_add(iv, m)


def ibp_vit(model: ViTTiny, x_l: torch.Tensor, x_u: torch.Tensor) -> Interval:
    """End-to-end IBP. x_l, x_u: (B, C, H, W).  Returns (B, num_classes) logit interval."""
    iv = Interval(x_l, x_u)
    iv = ibp_patch_embed(iv, model.patch_embed.proj)            # (B, N, E)
    pe = model.pos_embed                                          # (1, N, E)
    iv = Interval(iv.l + pe, iv.u + pe)
    for blk in model.blocks:
        iv = ibp_block(iv, blk)
    iv = ibp_rmsnorm(iv, model.norm)
    # mean-pool over tokens (linear, monotone)
    iv = Interval(iv.l.mean(dim=1), iv.u.mean(dim=1))             # (B, E)
    iv = ibp_linear_proj(iv, model.head)                          # (B, C)
    return iv

In [14]:
# ── Validation: bounds must contain the forward pass and shrink to zero at ε=0 ──
def load_vit(ckpt_path: Path) -> ViTTiny:
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ViTTiny(**payload['cfg']).to(device)
    sd = payload.get('state_dict_materialized', payload['state_dict'])
    sd = {k: v for k, v in sd.items() if 'parametrizations' not in k}
    model.load_state_dict(sd, strict=False)
    model.eval()
    return model


def get_eval_loader(n=20, batch_size=4, seed=1234):
    tf = transforms.ToTensor()
    test_ds = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True, transform=tf)
    rng = np.random.default_rng(seed)
    idx = sorted(rng.choice(len(test_ds), size=n, replace=False).tolist())
    subset = torch.utils.data.Subset(test_ds, idx)
    return torch.utils.data.DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=0)


@torch.no_grad()
def validate_ibp(model, loader, eps_list=(0.0, 0.01, 0.05)):
    """
    For each ε:
      • bounds should contain forward pass (l ≤ z(x) ≤ u, element-wise)
      • at ε=0 the gap (u-l) should be ~0 (modulo float)
      • count how many samples are IBP-certified
    """
    model.eval()
    out = {}
    for eps in eps_list:
        n_total = n_contain = n_cert = n_correct = 0
        max_gap = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            x_l = (x - eps).clamp(0, 1)
            x_u = (x + eps).clamp(0, 1)
            iv  = ibp_vit(model, x_l, x_u)
            z   = model(x)
            assert torch.all(iv.l - z <= 1e-3), f'lower not ≤ z (max viol={float((iv.l-z).max()):.3e})'
            assert torch.all(z - iv.u <= 1e-3), f'z not ≤ upper (max viol={float((z-iv.u).max()):.3e})'
            n_contain += len(y)
            max_gap = max(max_gap, float((iv.u - iv.l).max()))
            pred = z.argmax(1)
            n_correct += (pred == y).sum().item()
            # Certify: for true label y, lb_y > max_{c≠y} ub_c
            for i in range(len(y)):
                if pred[i] != y[i]: continue
                lb_y = iv.l[i, y[i]]
                ub_other = iv.u[i].clone(); ub_other[y[i]] = float('-inf')
                if lb_y > ub_other.max(): n_cert += 1
            n_total += len(y)
        out[eps] = dict(n=n_total, contain=n_contain, correct=n_correct,
                        certified=n_cert, max_gap=max_gap)
    return out

In [17]:
# ── Locate checkpoints (Drive first, local fallback) ────────────────────
def _find_ckpt(name: str) -> Path | None:
    candidates = [
        Path(f'/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_{name}/model.pt'),
        Path(f'runs/vit_tiny_{name}/model.pt'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

CKPTS = {n: _find_ckpt(n) for n in ('standard', 'lipmargin')}
for n, p in CKPTS.items():
    print(f'{n:10s} → {p}  ({"FOUND" if p else "MISSING"})')


standard   → /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard/model.pt  (FOUND)
lipmargin  → /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin/model.pt  (FOUND)


In [16]:
# ── Run validation on both checkpoints ────────────────────────────────────


loader = get_eval_loader(n=20, batch_size=4)

all_reports = {}
for name, path in CKPTS.items():
    if not path.exists():
        print(f'[skip] {name}: {path}'); continue
    print(f'\n══ {name} ══════════════════════════════')
    model = load_vit(path)
    rep   = validate_ibp(model, loader, eps_list=(0.0, 0.001, 0.005, 0.01, 0.03, 0.1))
    all_reports[name] = rep
    for eps, r in rep.items():
        print(f'  ε={eps:.3f}  contain={r["contain"]}/{r["n"]}  '
              f'correct={r["correct"]}/{r["n"]}  '
              f'IBP-cert={r["certified"]}/{r["n"]}  '
              f'max_gap={r["max_gap"]:.3e}')

# Save
out_path = Path('results/vit_p2'); out_path.mkdir(parents=True, exist_ok=True)
out_file = out_path / 'ibp_report.json'
out_file.write_text(json.dumps(all_reports, indent=2))
print(f'\nReport saved → {out_file}')


══ standard ══════════════════════════════


AssertionError: lower exceeds upper

In [ ]:
# ── Save report to local + Drive ─────────────────────────────────────────
import os, shutil
out = Path('results/vit_p2'); out.mkdir(parents=True, exist_ok=True)
(out / 'ibp_report.json').write_text(json.dumps(all_reports, indent=2))
print(f'Saved local → {out / "ibp_report.json"}')

drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
try:
    drive_out.mkdir(parents=True, exist_ok=True)
    (drive_out / 'ibp_report.json').write_text(json.dumps(all_reports, indent=2))
    print(f'Saved drive → {drive_out / "ibp_report.json"}')
except Exception as e:
    print(f'(skipped drive copy: {e})')
